# LeetCode 3414 — Maximum Score of Non-overlapping Intervals bằng Python

- Link: https://leetcode.com/problems/maximum-score-of-non-overlapping-intervals/
- Độ khó: **Hard**
- Chủ đề: **Quy hoạch động lập lịch có trọng số** (weighted interval scheduling) với **số khoảng tối đa 4**, sắp theo đầu phải + `bisect`, giá trị DP là **cặp (điểm, danh sách chỉ số đã sắp)** để phá hòa theo thứ tự từ điển

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maximumWeight(self, intervals: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Nắm DP lập lịch có trọng số: sắp theo **đầu phải**, với khoảng thứ `p` tìm khoảng cuối cùng **kết thúc trước** đầu trái của nó bằng `bisect`.
2. Thêm chiều "đã dùng bao nhiêu khoảng" (`≤ 4`).
3. Biến giá trị DP thành **cặp `(điểm, danh sách chỉ số đã sắp tăng)`** và so sánh: điểm **lớn hơn**, hòa thì danh sách **nhỏ hơn theo từ điển**.
4. Hiểu vì sao "tối ưu con" vẫn đúng với tiêu chí phụ từ điển (danh sách ≤ 4 phần tử, so sánh sau khi sắp).
5. Phân biệt "không giao nhau" **nghiêm ngặt**: hai khoảng chung một điểm biên là giao nhau.

## 2. Đề bài & ví dụ

`intervals[i] = [l, r, w]`. Chọn **tối đa 4** khoảng đôi một **không giao nhau** (không có điểm chung; chung đầu mút cũng tính là giao). Điểm = tổng trọng số. Trả về **mảng chỉ số nhỏ nhất theo thứ tự từ điển** trong các lựa chọn có điểm lớn nhất.

**Ví dụ 1:** `[[1,3,2],[4,5,2],[1,5,5],[6,9,3],[6,7,1],[8,9,1]]` → `[2,3]`.

**Ví dụ 2:** `[[5,8,1],[6,7,7],[4,7,3],[9,10,6],[7,8,2],[11,14,3],[3,5,5]]` → `[1,3,5,6]`.

**Ràng buộc:** `n ≤ 5·10^4`, `1 ≤ l ≤ r ≤ 10^9`, `1 ≤ w ≤ 10^9`.

In [1]:
from typing import List
import bisect, itertools

examples = [
    ([[1, 3, 2], [4, 5, 2], [1, 5, 5], [6, 9, 3], [6, 7, 1], [8, 9, 1]], [2, 3]),
    ([[5, 8, 1], [6, 7, 7], [4, 7, 3], [9, 10, 6], [7, 8, 2], [11, 14, 3], [3, 5, 5]], [1, 3, 5, 6]),
]
for iv, exp in examples:
    print(len(iv), 'khoảng -> kỳ vọng', exp)

6 khoảng -> kỳ vọng [2, 3]
7 khoảng -> kỳ vọng [1, 3, 5, 6]


## 3. Kiến thức nền

### 3.1. Không giao nhau nghiêm ngặt

`[l1, r1]` và `[l2, r2]` không giao nhau ⇔ `r1 < l2` hoặc `r2 < l1`.

In [2]:
def disjoint(a, b):
    return a[1] < b[0] or b[1] < a[0]

print(disjoint([1, 3], [4, 5]), disjoint([1, 3], [3, 5]), disjoint([6, 9], [1, 5]))

True False True


### 3.2. Lập lịch có trọng số cổ điển

Sắp theo `r`. `dp[p]` = tốt nhất trên `p` khoảng đầu. Khoảng thứ `p` (đã sắp) có đầu trái `l`: các khoảng tương thích phía trước là những khoảng có `r < l` — chính là **một tiền tố** của danh sách đã sắp, độ dài `j = bisect_left(R, l)`. `dp[p] = max(dp[p − 1], dp[j] + w)`.

### 3.3. So sánh cặp (điểm, danh sách)

Python so sánh danh sách theo từ điển sẵn (`[0, 5] < [1]`, `[1, 3] < [1, 3, 5]`). Ta coi `A` tốt hơn `B` nếu `A.điểm > B.điểm`, hoặc bằng điểm và `A.danh_sách < B.danh_sách`.

In [3]:
def better(a, b):
    if b is None:
        return True
    if a[0] != b[0]:
        return a[0] > b[0]
    return a[1] < b[1]

print(better((5, [0, 1]), (5, [2])), better((5, [2]), (5, [0, 1])), better((6, [9]), (5, [0])))

True False True


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: `[2]` (khoảng `[1,5]` nặng 5) + `[3]` (`[6,9]` nặng 3) = 8. Các lựa chọn khác: `{0, 1, 3}` = `2 + 2 + 3 = 7`; `{0, 1, 4, 5}` = `2 + 2 + 1 + 1 = 6`. Không có lựa chọn nào đạt 8 ngoài `{2, 3}` ⇒ `[2, 3]`.

Lưu ý `[1,3]` và `[4,5]` không chung điểm (3 < 4) nên chọn cùng được; nhưng `[1,5]` và `[6,9]`: 5 < 6 cũng được.

In [4]:
iv = examples[0][0]
for S in [(2, 3), (0, 1, 3), (0, 1, 4, 5)]:
    ok = all(disjoint(iv[a], iv[b]) for a, b in itertools.combinations(S, 2))
    print(S, 'hợp lệ' if ok else 'giao nhau', 'điểm', sum(iv[i][2] for i in S))

(2, 3) hợp lệ điểm 8
(0, 1, 3) hợp lệ điểm 7
(0, 1, 4, 5) hợp lệ điểm 6


## 5. Cách 1 — Brute Force: mọi tập ≤ 4 khoảng

In [5]:
def max_weight_brute(iv):
    best = None
    for c in range(1, 5):
        for S in itertools.combinations(range(len(iv)), c):
            if all(disjoint(iv[a], iv[b]) for a, b in itertools.combinations(S, 2)):
                key = (-sum(iv[i][2] for i in S), list(S))
                if best is None or key < best:
                    best = key
    return best[1]

for iv, exp in examples:
    print(max_weight_brute(iv), '| kỳ vọng', exp)

[2, 3] | kỳ vọng [2, 3]
[1, 3, 5, 6] | kỳ vọng [1, 3, 5, 6]


**Độ phức tạp:** `O(n^4)` — chỉ dùng cho `n ≤ 12`.

## 6. Ý tưởng thuật toán / bước đột phá

Sắp chỉ số theo `r`, `R` là danh sách đầu phải tương ứng. `dp[c][p]` = cặp tốt nhất dùng **tối đa `c`** khoảng trong `p` khoảng đầu (theo thứ tự đã sắp):

- `dp[0][p] = (0, [])`;
- `dp[c][p] = tốt hơn( dp[c][p − 1],  (dp[c−1][j].điểm + w,  sorted(dp[c−1][j].ds + [i])) )`, với `i` là chỉ số gốc của khoảng thứ `p`, `j = bisect_left(R, l_i)`.

Đáp án `dp[4][n].ds`. Mỗi trạng thái `O(log n + 4)` ⇒ `O(4 n log n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Cùng DP nhưng tìm `j` bằng quét tuyến tính (`O(n²)`)

In [6]:
def max_weight_quadratic(iv):
    n = len(iv)
    order = sorted(range(n), key=lambda i: iv[i][1])
    dp = [[(0, [])] * (n + 1) for _ in range(5)]
    for c in range(1, 5):
        for p in range(1, n + 1):
            i = order[p - 1]
            l, r, w = iv[i]
            j = 0
            while j < n and iv[order[j]][1] < l:
                j += 1
            prev = dp[c - 1][j]
            cand = (prev[0] + w, sorted(prev[1] + [i]))
            dp[c][p] = cand if better(cand, dp[c][p - 1]) else dp[c][p - 1]
    return dp[4][n][1]

for iv, exp in examples:
    print(max_weight_quadratic(iv), '| kỳ vọng', exp)

[2, 3] | kỳ vọng [2, 3]
[1, 3, 5, 6] | kỳ vọng [1, 3, 5, 6]


### 7.2. Cách 3 — Sắp theo đầu phải + `bisect` (lời giải chính)

In [7]:
def max_weight_dp(iv):
    n = len(iv)
    order = sorted(range(n), key=lambda i: iv[i][1])
    R = [iv[i][1] for i in order]
    dp = [[(0, [])] * (n + 1) for _ in range(5)]
    for c in range(1, 5):
        for p in range(1, n + 1):
            i = order[p - 1]
            l, r, w = iv[i]
            j = bisect.bisect_left(R, l)
            prev = dp[c - 1][j]
            cand = (prev[0] + w, sorted(prev[1] + [i]))
            dp[c][p] = cand if better(cand, dp[c][p - 1]) else dp[c][p - 1]
    return dp[4][n][1]

for iv, exp in examples:
    print(max_weight_dp(iv), '| kỳ vọng', exp)

[2, 3] | kỳ vọng [2, 3]
[1, 3, 5, 6] | kỳ vọng [1, 3, 5, 6]


### 7.3. Cách 4 — Đệ quy có nhớ, sắp theo đầu **trái**

`f(p, c)` = cặp tốt nhất chọn tối đa `c` khoảng trong các khoảng (đã sắp theo `l`) từ vị trí `p` trở đi. Lấy khoảng `p` ⇒ nhảy tới khoảng đầu tiên có `l > r_p` (`bisect_right`). Đệ quy theo `p` sâu tới `n` ⇒ chỉ dùng cho `n` vừa phải (hoặc tăng giới hạn đệ quy).

In [8]:
import sys
from functools import lru_cache

def max_weight_memo(iv):
    n = len(iv)
    order = sorted(range(n), key=lambda i: iv[i][0])
    L = [iv[i][0] for i in order]
    sys.setrecursionlimit(max(10000, 2 * n + 100))

    @lru_cache(maxsize=None)
    def f(p, c):
        if p == n or c == 0:
            return (0, ())
        best = f(p + 1, c)
        i = order[p]
        nxt = bisect.bisect_right(L, iv[i][1])
        sub = f(nxt, c - 1)
        cand = (sub[0] + iv[i][2], tuple(sorted(sub[1] + (i,))))
        if cand[0] > best[0] or (cand[0] == best[0] and list(cand[1]) < list(best[1])):
            best = cand
        return best

    res = list(f(0, 4)[1])
    f.cache_clear()
    return res

for iv, exp in examples:
    print(max_weight_memo(iv), '| kỳ vọng', exp)

[2, 3] | kỳ vọng [2, 3]
[1, 3, 5, 6] | kỳ vọng [1, 3, 5, 6]


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 1: in thứ tự sắp, chỉ số `j` tìm được và bảng `dp[c][p]` cuối mỗi hàng.

In [9]:
def trace(iv):
    n = len(iv)
    order = sorted(range(n), key=lambda i: iv[i][1])
    R = [iv[i][1] for i in order]
    print('thứ tự theo r:', order, '| R =', R)
    for p in range(1, n + 1):
        i = order[p - 1]
        print(f"  vị trí {p}: khoảng {i} = {iv[i]}, j = bisect_left(R, {iv[i][0]}) = {bisect.bisect_left(R, iv[i][0])}")
    dp = [[(0, [])] * (n + 1) for _ in range(5)]
    for c in range(1, 5):
        for p in range(1, n + 1):
            i = order[p - 1]; l, r, w = iv[i]
            prev = dp[c - 1][bisect.bisect_left(R, l)]
            cand = (prev[0] + w, sorted(prev[1] + [i]))
            dp[c][p] = cand if better(cand, dp[c][p - 1]) else dp[c][p - 1]
        print(f"c = {c}: dp = {dp[c][1:]}")
    print('đáp án:', dp[4][n][1])

trace(examples[0][0])

thứ tự theo r: [0, 1, 2, 4, 3, 5] | R = [3, 5, 5, 7, 9, 9]
  vị trí 1: khoảng 0 = [1, 3, 2], j = bisect_left(R, 1) = 0
  vị trí 2: khoảng 1 = [4, 5, 2], j = bisect_left(R, 4) = 1
  vị trí 3: khoảng 2 = [1, 5, 5], j = bisect_left(R, 1) = 0
  vị trí 4: khoảng 4 = [6, 7, 1], j = bisect_left(R, 6) = 3
  vị trí 5: khoảng 3 = [6, 9, 3], j = bisect_left(R, 6) = 3
  vị trí 6: khoảng 5 = [8, 9, 1], j = bisect_left(R, 8) = 4
c = 1: dp = [(2, [0]), (2, [0]), (5, [2]), (5, [2]), (5, [2]), (5, [2])]
c = 2: dp = [(2, [0]), (4, [0, 1]), (5, [2]), (6, [2, 4]), (8, [2, 3]), (8, [2, 3])]
c = 3: dp = [(2, [0]), (4, [0, 1]), (5, [2]), (6, [2, 4]), (8, [2, 3]), (8, [2, 3])]
c = 4: dp = [(2, [0]), (4, [0, 1]), (5, [2]), (6, [2, 4]), (8, [2, 3]), (8, [2, 3])]
đáp án: [2, 3]


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def maximumWeight(self, intervals: List[List[int]]) -> List[int]:
        n = len(intervals)
        order = sorted(range(n), key=lambda i: intervals[i][1])
        R = [intervals[i][1] for i in order]
        prev_row = [(0, [])] * (n + 1)
        for c in range(1, 5):
            row = [(0, [])] * (n + 1)
            for p in range(1, n + 1):
                i = order[p - 1]
                l, _, w = intervals[i]
                base = prev_row[bisect.bisect_left(R, l)]
                cand = (base[0] + w, sorted(base[1] + [i]))
                best = row[p - 1]
                if cand[0] > best[0] or (cand[0] == best[0] and cand[1] < best[1]):
                    best = cand
                row[p] = best
            prev_row = row
        return prev_row[n][1]

for iv, exp in examples:
    got = Solution().maximumWeight(iv)
    print(got, 'OK' if got == exp else 'SAI')

[2, 3] OK
[1, 3, 5, 6] OK


## 10. Vì sao thuật toán đúng

1. **Cấu trúc tối ưu:** xét `p` khoảng đầu (theo `r`). Lời giải tốt nhất hoặc không dùng khoảng thứ `p` (⇒ `dp[c][p − 1]`), hoặc dùng nó; khi đó các khoảng còn lại phải có `r < l_p` — là tiền tố `j` — và dùng tối đa `c − 1` khoảng.
2. **Tiêu chí phụ từ điển "tương thích" với DP:** gọi `T` là tập tối ưu (điểm lớn nhất, rồi danh sách sắp nhỏ nhất) có chứa khoảng `p`. Phần `T \ {p}` phải có điểm lớn nhất trong tiền tố `j` (nếu không, thay bằng phần tốt hơn sẽ tăng điểm). Giữa các phần cùng điểm, thêm cùng một phần tử `i` vào hai danh sách rồi sắp lại **bảo toàn thứ tự từ điển** giữa chúng (chèn cùng một số vào hai dãy đã sắp không đảo thứ tự so sánh — kiểm chứng bằng vét cạn ở mục 12). Vì vậy lưu "phần tốt nhất" ở mỗi trạng thái là đủ.
3. **"Tối đa `c`":** `dp[c][·]` khởi tạo bằng `(0, [])` và luôn so với `dp[c][p − 1]`, nên tự động bao gồm các lựa chọn ít khoảng hơn.
4. **Biên nghiêm ngặt:** `bisect_left(R, l)` đếm các khoảng có `r < l` — loại khoảng chạm biên.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi tập ≤ 4 | `O(n^4 · 16)` | `O(1)` |
| 2. DP + quét tuyến tính | `O(4 n²)` | `O(4 n)` |
| 3. DP + `bisect` | `O(4 n log n)` (+ sắp danh sách ≤ 4 phần tử) | `O(4 n)` |
| 4. Đệ quy có nhớ | `O(4 n log n)` | `O(4 n)` + ngăn xếp |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'quadratic': max_weight_quadratic, 'dp': max_weight_dp, 'memo': max_weight_memo,
         'submit': lambda iv: Solution().maximumWeight(iv)}
edge = [
    ([[1, 1, 5]], [0]),
    ([[1, 3, 1], [3, 5, 1]], [0]),                      # chạm biên: chỉ chọn được một, hòa -> chỉ số nhỏ
    ([[1, 2, 3], [3, 4, 3], [5, 6, 3], [7, 8, 3], [9, 10, 3]], [0, 1, 2, 3]),
    ([[1, 10, 5], [1, 2, 2], [3, 4, 3]], [0]),           # [0] (5) và [1,2] (5): [0] < [1, 2]
    ([[1, 2, 2], [3, 4, 3], [1, 10, 5]], [0, 1]),        # đổi chỗ: [0, 1] < [2]
]
for iv, exp in examples + edge:
    assert max_weight_brute(iv) == exp, (iv, max_weight_brute(iv))
    for name, f in impls.items():
        assert f(iv) == exp, (name, iv)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn (nhiều hòa điểm)

In [12]:
import random
rng = random.Random(3414)
cnt = 0
for _ in range(1500):
    n = rng.randint(1, 9)
    iv = []
    for _ in range(n):
        l = rng.randint(1, 10); iv.append([l, rng.randint(l, 12), rng.randint(1, 3)])
    exp = max_weight_brute(iv)
    for name, f in impls.items():
        assert f(iv) == exp, (name, iv)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ ngẫu nhiên.')

Khớp vét cạn trên 1500 bộ ngẫu nhiên.


### 12.3. Ca lớn `n = 5·10^4`

In [13]:
import time
n = 5 * 10 ** 4
def gen(n, span, wmax):
    out = []
    for _ in range(n):
        l = rng.randint(1, 10 ** 9 - span); out.append([l, l + rng.randint(0, span), rng.randint(1, wmax)])
    return out
cases = {'khoảng ngắn, trọng số lớn': gen(n, 10 ** 4, 10 ** 9), 'khoảng dài, trọng số 1 (nhiều hòa)': gen(n, 10 ** 8, 1),
         'giống hệt nhau': [[1, 5, 7]] * n}
for name, iv in cases.items():
    t = time.time(); a = Solution().maximumWeight(iv); t1 = time.time() - t
    t = time.time(); b = max_weight_dp(iv); t2 = time.time() - t
    assert a == b
    print(f"{name:34s}: {a} | nộp bài {t1:.2f}s | cách 3 {t2:.2f}s")

khoảng ngắn, trọng số lớn         : [383, 12555, 17042, 36052] | nộp bài 0.16s | cách 3 0.17s


khoảng dài, trọng số 1 (nhiều hòa): [0, 1, 3, 4] | nộp bài 0.19s | cách 3 0.22s
giống hệt nhau                    : [0] | nộp bài 0.07s | cách 3 0.09s


## 13. Các lỗi người mới hay gặp

Hàm tìm phản ví dụ (seed cố định):

In [14]:
def find_counterexample(wrong, seed=0, tries=20000):
    rng = random.Random(seed)
    best = None
    for _ in range(tries):
        n = rng.randint(1, 6)
        iv = []
        for _ in range(n):
            l = rng.randint(1, 8); iv.append([l, rng.randint(l, 9), rng.randint(1, 3)])
        if wrong(iv) != max_weight_brute(iv) and (best is None or len(iv) < len(best)):
            best = iv
    return best

def make_dp(strict=True, sort_list=True, tie_lex=True, only_four=False):
    def solve(iv):
        n = len(iv)
        order = sorted(range(n), key=lambda i: iv[i][1])
        R = [iv[i][1] for i in order]
        NEG = (-1, [])
        dp = [[(0, [])] * (n + 1)] + [[(0, []) if not only_four else NEG] * (n + 1) for _ in range(4)]
        for c in range(1, 5):
            for p in range(1, n + 1):
                i = order[p - 1]; l, r, w = iv[i]
                j = bisect.bisect_left(R, l) if strict else bisect.bisect_right(R, l)
                j = min(j, p - 1)
                prev = dp[c - 1][j]
                if prev[0] < 0:
                    dp[c][p] = dp[c][p - 1]; continue
                lst = prev[1] + [i]
                cand = (prev[0] + w, sorted(lst) if sort_list else lst)
                best = dp[c][p - 1]
                if cand[0] > best[0] or (tie_lex and cand[0] == best[0] and cand[1] < best[1]):
                    best = cand
                dp[c][p] = best
        return dp[4][n][1]
    return solve

### Lỗi 1 — Cho phép hai khoảng chung đầu mút (`r == l`)

In [15]:
wrong_touch = make_dp(strict=False)
ce = find_counterexample(wrong_touch, seed=1)
print('phản ví dụ:', ce, '| sai:', wrong_touch(ce), '| đúng:', max_weight_brute(ce))

phản ví dụ: [[7, 9, 1], [4, 7, 2]] | sai: [0, 1] | đúng: [1]


**Vì sao sai:** đề nói rõ chung biên là **giao nhau**; phải tìm các khoảng có `r < l` (dùng `bisect_left`).

### Lỗi 2 — Không sắp danh sách chỉ số (giữ theo thứ tự chọn)

In [16]:
wrong_unsorted = make_dp(sort_list=False)
print('sai :', wrong_unsorted(examples[1][0]), '| đúng:', Solution().maximumWeight(examples[1][0]))

sai : [6, 1, 3, 5] | đúng: [1, 3, 5, 6]


**Vì sao sai:** mảng chỉ số nhỏ nhất theo từ điển của một tập là mảng **đã sắp tăng**; so sánh hai lựa chọn cũng phải trên dạng đã sắp.

### Lỗi 3 — Hòa điểm thì giữ lựa chọn cũ (không xét thứ tự từ điển)

In [17]:
wrong_tie = make_dp(tie_lex=False)
ce = find_counterexample(wrong_tie, seed=3)
print('phản ví dụ:', ce, '| sai:', wrong_tie(ce), '| đúng:', max_weight_brute(ce))

phản ví dụ: [[2, 8, 2], [4, 5, 2]] | sai: [1] | đúng: [0]


**Vì sao sai:** nhiều lựa chọn cùng điểm tối đa; đề yêu cầu danh sách **nhỏ nhất theo từ điển**.

### Lỗi 4 — Bắt buộc chọn **đúng** 4 khoảng

In [18]:
wrong_exact4 = make_dp(only_four=True)
print('sai :', wrong_exact4(examples[0][0]), '| đúng:', Solution().maximumWeight(examples[0][0]))

sai : [0, 1, 4, 5] | đúng: [2, 3]


**Vì sao sai:** "tối đa 4" — ví dụ 1 chọn 2 khoảng cho điểm cao hơn mọi bộ 4 khoảng; thậm chí có thể không tồn tại 4 khoảng rời nhau.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Chọn các khoảng rời nhau tối đa trọng số ⇒ sắp theo đầu phải + `bisect` + DP.
- Giới hạn số khoảng nhỏ ⇒ thêm chiều `c`.
- Tiêu chí phụ "nhỏ nhất theo từ điển" với lời giải nhỏ (≤ 4 phần tử) ⇒ lưu luôn danh sách trong DP.
- Chú ý biên: "chung điểm là giao" ⇒ `r < l` nghiêm ngặt.

## 15. Biến thể của bài

1. **LC 1235 — Maximum Profit in Job Scheduling:** không giới hạn số khoảng, chỉ cần điểm.
2. **LC 2054 — Two Best Non-Overlapping Events:** tối đa 2 khoảng.
3. Tối đa `K` khoảng với `K` lớn ⇒ DP `O(K n log n)`; nếu chỉ cần điểm có thể dùng "Aliens trick" (phạt λ).
4. Khoảng chung biên được phép ⇒ đổi `bisect_left` thành `bisect_right`.

In [19]:
# Biến thể 1 (LC 1235): chỉ điểm, không giới hạn số khoảng; "chung biên được phép" như đề LC 1235
def job_scheduling(start, end, profit):
    jobs = sorted(zip(end, start, profit))
    E = [e for e, _, _ in jobs]
    dp = [0] * (len(jobs) + 1)
    for p, (e, s, w) in enumerate(jobs, 1):
        j = bisect.bisect_right(E, s)
        dp[p] = max(dp[p - 1], dp[j] + w)
    return dp[-1]

assert job_scheduling([1, 2, 3, 3], [3, 4, 5, 6], [50, 10, 40, 70]) == 120
assert job_scheduling([1, 2, 3, 4, 6], [3, 5, 10, 6, 9], [20, 20, 100, 70, 60]) == 150
print('LC 1235 ví dụ: 120, 150 — đúng.')

LC 1235 ví dụ: 120, 150 — đúng.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi tập ≤ 4 | `O(n^4)` | Oracle |
| 2 | DP + quét | `O(n²)` | Trung gian |
| 3 | DP + `bisect` | `O(n log n)` | **Nộp bài** |
| 4 | Đệ quy có nhớ theo `l` | `O(n log n)` | Góc nhìn top-down |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[[1,5,3],[6,10,3]]`.

**Bài 2 (tính tay):** `[[1,2,1],[2,3,1],[3,4,1],[4,5,1],[5,6,1]]`.

**Bài 3 (điền TODO):** viết `two_best_events(events)` (LC 2054: tối đa 2, chỉ trả điểm, chung biên là giao).

**Bài 4 (tự viết lại):** tổng quát `K` khoảng (tham số), trả về danh sách chỉ số.

**Bài 5 (mini challenge):** tối đa 4 khoảng nhưng trả về **số cách** chọn đạt điểm tối đa.

In [20]:
def two_best_events_todo(events):
    # TODO
    raise NotImplementedError

def max_weight_k_todo(intervals, K):
    # TODO
    raise NotImplementedError

def count_optimal_todo(intervals):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Hai khoảng rời nhau (5 < 6). (2) Chọn cả hai. (3) `[0, 1]`.

**Bài 2.** (1) Các khoảng kề chung biên ⇒ giao nhau. (2) Chọn được `0, 2, 4`. (3) Điểm 3.

**Bài 3.** (1) DP với `c ≤ 2`. (2) Chỉ lưu điểm. (3) Đối chiếu ví dụ LC 2054.

**Bài 4.** (1) Thay 4 bằng `K`. (2) Giữ hai hàng DP. (3) Đối chiếu vét cạn với `K` nhỏ.

**Bài 5.** (1) DP lưu `(điểm, số cách)` cho mỗi `(c, p)`. (2) Cẩn thận: "tối đa `c`" cần đếm theo **đúng** số khoảng rồi cộng. (3) Vét cạn kiểm tra.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
a1 = Solution().maximumWeight([[1, 5, 3], [6, 10, 3]])
a2 = Solution().maximumWeight([[1, 2, 1], [2, 3, 1], [3, 4, 1], [4, 5, 1], [5, 6, 1]])
assert a1 == [0, 1] and a2 == max_weight_brute([[1, 2, 1], [2, 3, 1], [3, 4, 1], [4, 5, 1], [5, 6, 1]]) == [0, 2, 4]
print('Bài 1:', a1, '| Bài 2:', a2)

def two_best_events(events):
    order = sorted(range(len(events)), key=lambda i: events[i][1])
    R = [events[i][1] for i in order]
    best1 = [0] * (len(order) + 1)
    ans = 0
    for p in range(1, len(order) + 1):
        s, e, v = events[order[p - 1]]
        best1[p] = max(best1[p - 1], v)
        ans = max(ans, v + best1[bisect.bisect_left(R, s)])
    return ans

assert two_best_events([[1, 3, 2], [4, 5, 2], [2, 4, 3]]) == 4
assert two_best_events([[1, 3, 2], [4, 5, 2], [1, 5, 5]]) == 5
assert two_best_events([[1, 5, 3], [1, 5, 1], [6, 6, 5]]) == 8
print('Bài 3: LC 2054 ví dụ -> 4, 5, 8')

Bài 1: [0, 1] | Bài 2: [0, 2, 4]
Bài 3: LC 2054 ví dụ -> 4, 5, 8


In [22]:
# Bài 4, 5
def max_weight_k(iv, K):
    n = len(iv)
    order = sorted(range(n), key=lambda i: iv[i][1])
    R = [iv[i][1] for i in order]
    prev = [(0, [])] * (n + 1)
    for c in range(1, K + 1):
        row = [(0, [])] * (n + 1)
        for p in range(1, n + 1):
            i = order[p - 1]; l, _, w = iv[i]
            base = prev[bisect.bisect_left(R, l)]
            cand = (base[0] + w, sorted(base[1] + [i]))
            row[p] = cand if better(cand, row[p - 1]) else row[p - 1]
        prev = row
    return prev[n][1]

def brute_k(iv, K):
    best = (0, [])
    for c in range(1, K + 1):
        for S in itertools.combinations(range(len(iv)), c):
            if all(disjoint(iv[a], iv[b]) for a, b in itertools.combinations(S, 2)):
                cand = (sum(iv[i][2] for i in S), list(S))
                if better(cand, best):
                    best = cand
    return best[1]

def count_optimal(iv):
    n = len(iv)
    best_score, ways = 0, 0
    for c in range(1, 5):
        for S in itertools.combinations(range(n), c):
            if all(disjoint(iv[a], iv[b]) for a, b in itertools.combinations(S, 2)):
                sc = sum(iv[i][2] for i in S)
                if sc > best_score: best_score, ways = sc, 1
                elif sc == best_score: ways += 1
    return best_score, ways

rng = random.Random(19)
for _ in range(400):
    n = rng.randint(1, 8); iv = []
    for _ in range(n):
        l = rng.randint(1, 10); iv.append([l, rng.randint(l, 12), rng.randint(1, 3)])
    K = rng.randint(1, 5)
    assert max_weight_k(iv, K) == brute_k(iv, K)
    assert max_weight_k(iv, 4) == Solution().maximumWeight(iv)
print('Bài 4 khớp vét cạn. Bài 5 (vét cạn) ví dụ 1 -> (điểm, số cách) =', count_optimal(examples[0][0]))

Bài 4 khớp vét cạn. Bài 5 (vét cạn) ví dụ 1 -> (điểm, số cách) = (8, 1)


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
order = sắp chỉ số theo r; R = [r theo thứ tự]
prev = [(0, [])]*(n+1)
for c in 1..4:
    row[p] = tốt_hơn(row[p-1], (prev[j].điểm + w, sorted(prev[j].ds + [i])))   # j = bisect_left(R, l)
    prev = row
return prev[n].ds
tốt_hơn: điểm lớn hơn, hòa thì danh sách nhỏ hơn
```

**Checklist:**

- [ ] `bisect_left` (chung biên là giao)?
- [ ] Danh sách chỉ số luôn được **sắp** trước khi so sánh/trả về?
- [ ] Hòa điểm ⇒ so sánh từ điển?
- [ ] "Tối đa 4" (bao gồm lựa chọn ít khoảng hơn)?
- [ ] Trả về **chỉ số gốc**, không phải vị trí sau khi sắp?